# Polyp Segmentation with MONAI - Kvasir-SEG
**Project 22**: Polyp segmentation from colonoscopy images using MONAI UNet, Kvasir-SEG dataset with Dice/IoU metrics and boundary visualization.

In [1]:
import os
import sys
import random
import numpy as np
import json
import zipfile
import shutil
import requests
import io
import ssl
import warnings
warnings.filterwarnings('ignore')

# PyTorch & MONAI
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
import monai
from monai.data import DataLoader as MonaiDataLoader, Dataset as MonaiDataset
from monai.transforms import (
    Compose, LoadImageD, ToTensorD, EnsureChannelFirstD,
    ResizeD, RandFlipd, RandRotate90d, RandAffineD, RandGaussianNoised,
    NormalizeIntensityd, EnsureTyped
)
from monai.networks.nets import UNet
from monai.losses import DiceLoss
from monai.metrics import DiceMetric

# Vision
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# Utils
from scipy import ndimage
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print(f"MONAI: {monai.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

e:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


PyTorch: 2.6.0+cu124
MONAI: 1.5.2
CUDA: True


In [2]:
# Setup
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Paths
current_dir = Path.cwd()
REPO_ROOT = Path(current_dir).parents[2] if 'Computer Vision' in str(current_dir) else Path.cwd()
DATA_DIR = os.path.join(str(REPO_ROOT), 'data', 'Kvasir-SEG')
SAVE_DIR = str(Path.cwd())

os.makedirs(DATA_DIR, exist_ok=True)
dataset_root = os.path.join(DATA_DIR, 'kvasir-seg')

# Device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
torch.cuda.manual_seed_all(SEED) if torch.cuda.is_available() else None
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Device: {device}")
print(f"Save dir: {SAVE_DIR}")
print(f"Data dir: {dataset_root}")

Device: cuda:0
Save dir: e:\Github\Machine-Learning-Projects\Computer Vision\Polyp Lesion Segmentation\Source Code
Data dir: e:\Github\Machine-Learning-Projects\data\Kvasir-SEG\kvasir-seg


In [3]:
# Config
CONFIG = {
    'image_size': (352, 352),
    'batch_size': 16,
    'num_epochs': 30,  # Reduced for faster demo
    'learning_rate': 1e-3,
    'val_split': 0.15,
    'test_split': 0.15,
}

print("Configuration:", CONFIG)

Configuration: {'image_size': (352, 352), 'batch_size': 16, 'num_epochs': 30, 'learning_rate': 0.001, 'val_split': 0.15, 'test_split': 0.15}


In [4]:
# Download Kvasir-SEG if needed
if not os.path.exists(os.path.join(dataset_root, 'images')):
    print("Downloading Kvasir-SEG dataset...")
    url = "https://datasets.simula.no/downloads/kvasir-seg.zip"
    
    try:
        response = requests.get(url, stream=True, timeout=60, verify=False)
        response.raise_for_status()
        
        zip_buffer = io.BytesIO()
        total_size = 0
        for chunk in response.iter_content(8192):
            if chunk:
                zip_buffer.write(chunk)
                total_size += len(chunk)
                print(f"\rDownloaded: {total_size / (1024*1024):.1f} MB", end='', flush=True)
        
        zip_buffer.seek(0)
        print(f"\n✓ Download complete")
        
        print("Extracting...")
        os.makedirs(dataset_root, exist_ok=True)
        with zipfile.ZipFile(zip_buffer, 'r') as zf:
            zf.extractall(dataset_root)
        
        # Move nested contents
        extracted = os.listdir(dataset_root)
        for item in extracted:
            if os.path.isdir(os.path.join(dataset_root, item)) and item != 'images' and item != 'masks':
                nested_path = os.path.join(dataset_root, item)
                for nested in os.listdir(nested_path):
                    src = os.path.join(nested_path, nested)
                    dst = os.path.join(dataset_root, nested)
                    if os.path.exists(dst):
                        shutil.rmtree(dst)
                    shutil.move(src, dst)
                shutil.rmtree(nested_path)
        
        print(f"✓ Dataset ready")
    except Exception as e:
        print(f"Download failed: {e}")
        raise
else:
    print(f"✓ Dataset already exists")

num_imgs = len(os.listdir(os.path.join(dataset_root, 'images')))
num_masks = len(os.listdir(os.path.join(dataset_root, 'masks')))
print(f"Images: {num_imgs}, Masks: {num_masks}")

✓ Dataset already exists
Images: 1000, Masks: 1000


In [16]:
# Create dataset
img_dir = os.path.join(dataset_root, 'images')
mask_dir = os.path.join(dataset_root, 'masks')

img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg'))])
mask_files = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

data_list = []
for img_name in img_files:
    if os.path.exists(os.path.join(mask_dir, img_name)):
        data_list.append({
            'image': os.path.join(img_dir, img_name),
            'label': os.path.join(mask_dir, img_name)
        })

print(f"Total pairs: {len(data_list)}")

# Split
n_total = len(data_list)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)
n_test = n_total - n_train - n_val

indices = list(range(n_total))
random.shuffle(indices)

train_data = [data_list[i] for i in indices[:n_train]]
val_data = [data_list[i] for i in indices[n_train:n_train+n_val]]
test_data = [data_list[i] for i in indices[n_train+n_val:]]

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

Total pairs: 1000
Train: 700, Val: 150, Test: 150


In [18]:
# Custom dataset class with preprocessing
class PolypDataset(torch.utils.data.Dataset):
    def __init__(self, data_list, is_train=False, image_size=(352, 352)):
        self.data_list = data_list
        self.is_train = is_train
        self.image_size = image_size
    
    def __len__(self):
        return len(self.data_list)
    
    def __getitem__(self, idx):
        data_dict = self.data_list[idx]
        img_path = data_dict['image']
        mask_path = data_dict['label']
        
        # Load images
        img = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')
        
        img_arr = np.array(img, dtype=np.float32)
        mask_arr = np.array(mask, dtype=np.float32) / 255.0
        
        # Resize
        img_resized = cv2.resize(img_arr, self.image_size, interpolation=cv2.INTER_LINEAR)
        mask_resized = cv2.resize(mask_arr, self.image_size, interpolation=cv2.INTER_NEAREST)
        
        # Normalize image to [-1, 1]
        img_normalized = (img_resized / 127.5) - 1.0
        
        # Data augmentation for training
        if self.is_train:
            # Random flip
            if np.random.rand() > 0.5:
                img_normalized = np.flip(img_normalized, axis=0).copy()
                mask_resized = np.flip(mask_resized, axis=0).copy()
            if np.random.rand() > 0.5:
                img_normalized = np.flip(img_normalized, axis=1).copy()
                mask_resized = np.flip(mask_resized, axis=1).copy()
        
        # Convert to CHW
        img_tensor = torch.from_numpy(np.transpose(img_normalized, (2, 0, 1))).float()
        mask_tensor = torch.from_numpy(mask_resized[np.newaxis, :, :]).float()
        
        return {'image': img_tensor, 'label': mask_tensor}

# Create datasets
train_ds = PolypDataset(train_data, is_train=True, image_size=CONFIG['image_size'])
val_ds = PolypDataset(val_data, is_train=False, image_size=CONFIG['image_size'])
test_ds = PolypDataset(test_data, is_train=False, image_size=CONFIG['image_size'])

# Create loaders
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)

print(f"✓ Loaders ready: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}")

✓ Loaders ready: train=44, val=10, test=10


In [19]:
# Model
model = UNet(
    spatial_dims=2,
    in_channels=3,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm='instance',
    dropout=0.2
).to(device)

dice_loss = DiceLoss(to_onehot_y=False, sigmoid=True)
ce_loss = nn.BCEWithLogitsLoss()
dice_metric = DiceMetric(include_background=True, reduction='mean_batch', get_not_nans=True)

optimizer = Adam(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

params = sum(p.numel() for p in model.parameters())
print(f"✓ Model: {params / 1e6:.2f}M parameters")

✓ Model: 1.63M parameters


In [21]:
# Training
def train_epoch(model, loader, optimizer, device):
    model.train()
    total = 0.0
    for batch in loader:
        img, mask = batch['image'].to(device), batch['label'].to(device)
        logits = model(img)
        dice_l = dice_loss(torch.sigmoid(logits), mask)
        ce_l = ce_loss(logits, mask)
        loss = 0.7 * dice_l + 0.3 * ce_l
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)

def val_epoch(model, loader, device):
    model.eval()
    dice_scores = []
    with torch.no_grad():
        for batch in loader:
            img, mask = batch['image'].to(device), batch['label'].to(device)
            logits = model(img)
            pred = torch.sigmoid(logits)
            
            # Compute Dice manually
            for i in range(pred.shape[0]):
                p = (pred[i, 0] > 0.5).float()
                m = mask[i, 0]
                intersection = torch.sum(p * m)
                dice = 2 * intersection / (torch.sum(p) + torch.sum(m) + 1e-7)
                dice_scores.append(dice.cpu().item())
    
    return np.mean(dice_scores) if dice_scores else 0.0

print(f"Training for {CONFIG['num_epochs']} epochs...")
train_losses, val_dices, best_val_dice = [], [], 0.0
best_path = os.path.join(SAVE_DIR, 'best_polyp_model.pth')
best_epoch = 0

for epoch in range(CONFIG['num_epochs']):
    tloss = train_epoch(model, train_loader, optimizer, device)
    vdice = val_epoch(model, val_loader, device)
    train_losses.append(tloss)
    val_dices.append(vdice)
    scheduler.step(vdice)
    
    if vdice > best_val_dice:
        best_val_dice = vdice
        best_epoch = epoch
        torch.save(model.state_dict(), best_path)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}: Loss={tloss:.4f}, ValDice={vdice:.4f}")

print(f"\n✓ Best Dice: {best_val_dice:.4f} at epoch {best_epoch+1}")
model.load_state_dict(torch.load(best_path, map_location=device))

Training for 30 epochs...
Epoch   1: Loss=0.7171, ValDice=0.4460
Epoch   5: Loss=0.6914, ValDice=0.5490
Epoch  10: Loss=0.6651, ValDice=0.5861
Epoch  15: Loss=0.6369, ValDice=0.6369
Epoch  20: Loss=0.6132, ValDice=0.7397
Epoch  25: Loss=0.5987, ValDice=0.7712
Epoch  30: Loss=0.5840, ValDice=0.7655

✓ Best Dice: 0.7809 at epoch 29


<All keys matched successfully>

In [23]:
# Evaluate on test set
print("Evaluating on test set...")

model.eval()
dice_scores, iou_scores = [], []

with torch.no_grad():
    for batch in test_loader:
        img, mask = batch['image'].to(device), batch['label'].to(device)
        logits = model(img)
        pred = torch.sigmoid(logits).cpu().numpy()
        mask_np = mask.cpu().numpy()
        
        for i in range(pred.shape[0]):
            p = (pred[i, 0] > 0.5).astype(np.uint8)
            m = (mask_np[i, 0] > 0.5).astype(np.uint8)
            
            dice = 2 * np.sum(p * m) / (np.sum(p) + np.sum(m) + 1e-7)
            intersection = np.sum(p * m)
            union = np.sum(p) + np.sum(m) - intersection
            iou = intersection / (union + 1e-7)
            
            dice_scores.append(dice)
            iou_scores.append(iou)

print(f"✓ Test Dice: {np.mean(dice_scores):.4f} ± {np.std(dice_scores):.4f}")
print(f"✓ Test IoU:  {np.mean(iou_scores):.4f} ± {np.std(iou_scores):.4f}")

Evaluating on test set...
✓ Test Dice: 0.7667 ± 0.2145
✓ Test IoU:  0.6635 ± 0.2435


In [25]:
# Export metrics
metrics = {
    'model': 'UNet',
    'dataset': 'Kvasir-SEG',
    'image_size': CONFIG['image_size'],
    'batch_size': CONFIG['batch_size'],
    'num_epochs': CONFIG['num_epochs'],
    'learning_rate': CONFIG['learning_rate'],
    'train_size': len(train_data),
    'val_size': len(val_data),
    'test_size': len(test_data),
    'best_epoch': best_epoch + 1,
    'best_val_dice': float(best_val_dice),
    'test_dice_mean': float(np.mean(dice_scores)),
    'test_dice_std': float(np.std(dice_scores)),
    'test_iou_mean': float(np.mean(iou_scores)),
    'test_iou_std': float(np.std(iou_scores)),
}

metrics_path = os.path.join(SAVE_DIR, 'metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"✓ Metrics saved to metrics.json")
print(json.dumps(metrics, indent=2))

✓ Metrics saved to metrics.json
{
  "model": "UNet",
  "dataset": "Kvasir-SEG",
  "image_size": [
    352,
    352
  ],
  "batch_size": 16,
  "num_epochs": 30,
  "learning_rate": 0.001,
  "train_size": 700,
  "val_size": 150,
  "test_size": 150,
  "best_epoch": 29,
  "best_val_dice": 0.7809417302698906,
  "test_dice_mean": 0.7667162492007307,
  "test_dice_std": 0.21448196172838493,
  "test_iou_mean": 0.6635063182584712,
  "test_iou_std": 0.24349027559567238
}


In [26]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, label='Train Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(val_dices, label='Val Dice')
ax2.axvline(best_epoch, color='r', linestyle='--', label=f'Best (epoch {best_epoch+1})')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Dice')
ax2.set_title('Validation Dice')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=100, bbox_inches='tight')
print("✓ Training curves saved")

✓ Training curves saved


In [27]:
# Boundary visualization
def extract_boundaries(mask, threshold=0.5):
    binary = (mask > threshold).astype(np.uint8) * 255
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return contours

print("\nGenerating boundary visualizations...")
fig, axes = plt.subplots(6, 4, figsize=(16, 24))
model.eval()

with torch.no_grad():
    for idx in range(6):
        if idx < len(test_data):
            sample = test_data[idx]
            img = Image.open(sample['image']).convert('RGB')
            mask_gt = Image.open(sample['label']).convert('L')
            
            # Infer
            img_arr = np.array(img, dtype=np.float32)
            img_resized = cv2.resize(img_arr, CONFIG['image_size'])
            img_norm = (img_resized / 127.5) - 1.0
            img_tensor = torch.from_numpy(np.transpose(img_norm, (2, 0, 1))).unsqueeze(0).to(device)
            
            logits = model(img_tensor)
            pred_prob = torch.sigmoid(logits).squeeze().cpu().numpy()
            
            # Resize back
            pred_resized = cv2.resize(pred_prob, (img_arr.shape[1], img_arr.shape[0]))
            
            # GT
            mask_arr = np.array(mask_gt, dtype=np.float32) / 255.0
            
            # Boundaries
            pred_contours = extract_boundaries(pred_resized)
            gt_contours = extract_boundaries(mask_arr)
            
            # Draw on original
            img_pred = img_arr.astype(np.uint8).copy()
            img_gt = img_arr.astype(np.uint8).copy()
            img_both = img_arr.astype(np.uint8).copy()
            
            cv2.drawContours(img_pred, pred_contours, -1, (0, 0, 255), 2)
            cv2.drawContours(img_gt, gt_contours, -1, (0, 255, 0), 2)
            cv2.drawContours(img_both, pred_contours, -1, (0, 0, 255), 2)
            cv2.drawContours(img_both, gt_contours, -1, (0, 255, 0), 2)
            
            # Plot
            dice = compute_dice(pred_resized, mask_arr)
            axes[idx, 0].imshow(cv2.cvtColor(img_pred, cv2.COLOR_BGR2RGB))
            axes[idx, 0].set_title(f'Pred (Red)\nDice: {dice:.3f}')
            axes[idx, 0].axis('off')
            
            axes[idx, 1].imshow(cv2.cvtColor(img_gt, cv2.COLOR_BGR2RGB))
            axes[idx, 1].set_title('GT (Green)')
            axes[idx, 1].axis('off')
            
            axes[idx, 2].imshow(cv2.cvtColor(img_both, cv2.COLOR_BGR2RGB))
            axes[idx, 2].set_title('Overlay')
            axes[idx, 2].axis('off')
            
            axes[idx, 3].imshow(pred_resized, cmap='hot')
            axes[idx, 3].set_title('Heatmap')
            axes[idx, 3].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'boundary_visualization.png'), dpi=100, bbox_inches='tight')
print("✓ Boundary visualization saved")


Generating boundary visualizations...
✓ Boundary visualization saved


In [29]:
# Summary and artifact verification
print("\n" + "="*60)
print("PROJECT SUMMARY: Polyp Lesion Segmentation")
print("="*60)

artifacts_found = []
for fname in os.listdir(SAVE_DIR):
    if fname.endswith(('.pth', '.json', '.png')):
        artifacts_found.append(fname)

summary = f"""
CONFIGURATION:
  Model: UNet (1.63M parameters)
  Dataset: Kvasir-SEG
  Image Size: {CONFIG['image_size']}
  Batch Size: {CONFIG['batch_size']}
  Epochs: {CONFIG['num_epochs']}
  Best Val Dice: {best_val_dice:.4f}
  Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}

TEST RESULTS:
  Dice: {np.mean(dice_scores):.4f} ± {np.std(dice_scores):.4f}
  IoU:  {np.mean(iou_scores):.4f} ± {np.std(iou_scores):.4f}
  Samples: {len(dice_scores)}

ARTIFACTS SAVED:
"""
for artifact in sorted(artifacts_found):
    summary += f"  - {artifact}\n"

print(summary)
print("="*60)


PROJECT SUMMARY: Polyp Lesion Segmentation

CONFIGURATION:
  Model: UNet (1.63M parameters)
  Dataset: Kvasir-SEG
  Image Size: (352, 352)
  Batch Size: 16
  Epochs: 30
  Best Val Dice: 0.7809
  Train: 700, Val: 150, Test: 150

TEST RESULTS:
  Dice: 0.7667 ± 0.2145
  IoU:  0.6635 ± 0.2435
  Samples: 150

ARTIFACTS SAVED:
  - best_polyp_model.pth
  - boundary_visualization.png
  - eda_samples.png
  - metrics.json
  - training_curves.png

